# AutoTune Studio on Kaggle

Fine-tune an LLM (LoRA / QLoRA) from a browser UI that opens in a **separate tab**. Datasets are streamed, so you never configure a row count.

> **Status: UNVERIFIED.** This notebook has **not been executed on Kaggle yet**. It is written
> against the implemented `Studio` API (verified locally on CPU), but installation, GPU training
> (Unsloth / QLoRA) and the cloudflared tunnel path have not been run on a real Kaggle VM.
> Please report what happens.

**Before you start (notebook *Settings* panel on the right):**

* *Accelerator*: **GPU T4 x2** or **GPU P100**. The studio trains on one GPU.
* *Internet*: **ON**. This needs a **phone-verified** Kaggle account. Without internet, pip, the Hugging Face Hub and the tunnel that serves the UI do not work.
* Use an interactive session. A *Save & Run All* (batch) run has no browser to open the UI.

**Security:** `studio.launch()` displays a link whose `#token=...` fragment is a session credential.
Anyone who has that link can control the studio while this kernel runs.
**Clear all outputs before you share, commit or publish this notebook**
(*Edit > Clear All Outputs*; also before *Save Version*, which stores outputs), and run the shutdown cell when you are done.

## 2. Check the GPU

In [ ]:
!nvidia-smi || echo 'No NVIDIA GPU visible: Settings > Accelerator > GPU (QLoRA needs CUDA).'

## 3. Install

There is no PyPI release yet. Pick one source:

* **(a)** set `AUTOTUNE_SOURCE` to a git URL of the repository. The repository includes the built browser UI, so a git install has it too (checked below). The Python API still works.
* **(b)** leave `AUTOTUNE_SOURCE` empty and add the wheel as a Kaggle dataset: *Datasets > New Dataset*, upload `autotune_studio-<version>-py3-none-any.whl`, then in this notebook *Add Input* and pick that dataset. The cell searches `/kaggle/input` for the wheel. The wheel includes the UI.

The dependencies still come from PyPI, so Internet must be ON. The preinstalled torch is kept (no reinstall). **Unsloth's install instructions change often:** if the GPU-extras step fails, follow https://github.com/unslothai/unsloth for the current Kaggle command. Without Unsloth, training falls back to the Hugging Face backend (QLoRA still works with `bitsandbytes`).

In [ ]:
AUTOTUNE_SOURCE = ""        # e.g. "https://github.com/<owner>/autotune-studio.git"; empty = wheel from /kaggle/input
INSTALL_GPU_EXTRAS = True   # unsloth, bitsandbytes, nvidia-ml-py

import glob, subprocess, sys

def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

if AUTOTUNE_SOURCE:
    pip_install(f"autotune-studio[hf,train] @ git+{AUTOTUNE_SOURCE}")
else:
    wheels = sorted(glob.glob("/kaggle/input/**/autotune_studio-*.whl", recursive=True))
    if not wheels:
        raise SystemExit("No autotune_studio-*.whl under /kaggle/input. Add the wheel dataset as an input "
                         "or set AUTOTUNE_SOURCE.")
    print("installing", wheels[-1])
    pip_install(f"{wheels[-1]}[hf,train]")

if INSTALL_GPU_EXTRAS:
    try:
        pip_install("unsloth", "bitsandbytes", "nvidia-ml-py")
    except subprocess.CalledProcessError:
        print("GPU extras failed to install. See https://github.com/unslothai/unsloth for the current "
              "install command; the studio still runs with the Hugging Face backend.")

In [ ]:
# The browser UI ships as built files (committed in the repository and included in the wheel).
# does not contain them (they are build output), and then only the Python API works.
import importlib.resources as _res
_ui = _res.files("autotune_studio") / "web" / "index.html"
print("Browser UI bundled:", _ui.is_file())
if not _ui.is_file():
    print("The UI is missing: install the wheel (option b) or build the frontend before installing.")

## 4. Optional: Hugging Face token

Needed only for gated or private models/datasets. *Add-ons > Secrets*: add `HF_TOKEN` and attach it to this notebook. The token is never printed.

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    _tok = UserSecretsClient().get_secret("HF_TOKEN")
    if _tok:
        os.environ["HF_TOKEN"] = _tok  # read by huggingface_hub / transformers
    print("HF token:", "found in Kaggle secrets" if _tok else "not set")
except Exception as exc:  # secret missing or not attached
    print(f"HF token: not available ({type(exc).__name__}); only needed for gated/private repos")
finally:
    _tok = None

## 5. Create the studio and detect the runtime

Outputs go to `/kaggle/working/autotune` (kept with a saved notebook version, about 20 GB). `/kaggle/input` is read-only. On Kaggle the studio probes outbound internet: if `internet: NO` is printed, switch Internet ON in the settings and restart the session.

In [ ]:
from autotune_studio import Studio

studio = Studio()
env = studio.detect_environment()
hw = studio.detect_hardware()

print("runtime:", env.kind.value, "| python", env.python_version)
print("workspace:", env.workspace_dir)
print("allowed dataset roots:", env.allowed_roots)
print("internet:", {None: "not probed", True: "yes", False: "NO"}[env.internet_access])
print("CUDA:", hw.cuda_available, "| CUDA version:", hw.cuda_version, "| torch:", hw.torch_version,
      "| bf16:", hw.bf16_supported)
for g in hw.gpus:
    print(f"GPU {g.index}: {g.name}, {g.total_vram_bytes / 2**30:.1f} GiB VRAM")
print(f"RAM: {hw.ram_total_bytes / 2**30:.1f} GiB | disk free: {hw.disk_free_bytes / 2**30:.1f} GiB")
for note in env.notes + hw.notes:
    print("  note:", note)

## 6. Launch the UI

Kaggle has no built-in port proxy, so the studio starts a **cloudflared quick tunnel** (the official binary, pinned version, SHA256-verified on download) and then checks `GET <tunnel>/api/health` through it. `publicly_verified=True` means that probe succeeded. Quick tunnels have no uptime guarantee. If the tunnel fails, the backend keeps running and the diagnostics say why (Internet off, download blocked, no URL in time). See `docs/kaggle.md`. Click the button; the UI opens in a new tab.

In [ ]:
info = studio.launch()  # shows an "Open AutoTune Studio" button; the cell returns immediately

print("method:", info.method, "| public access verified:", info.publicly_verified)
print("backend (inside the VM):", info.local_url)
if info.public_url:
    print("public URL (without token):", info.public_url)
for d in info.diagnostics:
    print("  -", d)
# Do not print info.open_url: it contains the session token.

## 7. Optional: the same workflow from pure Python

You do not need the browser UI. This cell trains a small model on a Hugging Face dataset.
Set `RUN_PYTHON_WORKFLOW = True` to run it.

* **Zero row configuration:** you never pass a row count. The dataset is streamed. If its size is
  unknown (`rows: None`), the studio cannot turn epochs into steps, so you **must** set `max_steps`
  (otherwise `start_training()` raises `StepsRequired`). `max_steps=60` is a short smoke run.
* `method="qlora"` needs a CUDA GPU plus `unsloth` or `bitsandbytes` (installed in step 3).
* Only one training job runs at a time. You can watch it live in the UI tab while this cell waits.

In [ ]:
RUN_PYTHON_WORKFLOW = False  # set to True to train from Python

if RUN_PYTHON_WORKFLOW:
    ds = studio.load_dataset("huggingface", name="yahma/alpaca-cleaned")
    print("dataset:", ds.dataset_id, "| format:", ds.detected_format,
          "| rows:", ds.cardinality.rows, "(None = unknown, max_steps required)")

    studio.configure_model("unsloth/Llama-3.2-1B-Instruct")
    studio.configure_training(method="qlora", max_steps=60)
    steps, warnings = studio.validate_training()
    print("planned steps:", steps)
    for w in warnings:
        print("  warning:", w)

    studio.start_training()
    status = studio.wait()  # blocks until the job finishes
    print("state:", status.state.value, "| step:", status.step, "| loss:", status.loss)
    if status.error:
        print("error:", status.error)
    else:
        result = studio.export_model()  # LoRA adapter from the latest checkpoint
        print("exported", result.format, "to", result.path, f"({result.size_bytes / 2**20:.1f} MiB)")
        # Merged 16-bit model instead: studio.export_model(format="merged_16bit")

## 8. Shut down

In [ ]:
# Stops training (a checkpoint is saved first), the tunnel, the web server and the resource sampler.
studio.shutdown()
print("AutoTune Studio stopped.")